In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"
MAX_LEN = 128


# =========================
# LOAD DATA
# =========================
train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")


# =========================
# CLEAN
# =========================
def clean(x):
    return str(x).lower()

for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)


# =========================
# PARSE word_id
# =========================
def parse_ids(df):
    df["word_pos"] = df["word_id"].str.extract(r"_(\d+)$").astype(int)
    df["page"] = df["word_id"].str.extract(r"page_(\d+)_").astype(int)
    df["doc"] = df["word_id"].str.extract(r"^(.*?)_\d+_page")
    return df

train = parse_ids(train)
test = parse_ids(test)


# =========================
# NORMALIZE TRT (paper)
# =========================
scaler_y = StandardScaler()
train["answer"] = scaler_y.fit_transform(train[["answer"]])


# =========================
# SPLIT (NO LEAKAGE)
# =========================
docs = train["doc"].unique()
train_docs, val_docs = train_test_split(docs, test_size=0.1, random_state=42)

train_df = train[train["doc"].isin(train_docs)]
val_df = train[train["doc"].isin(val_docs)]


# =========================
# BUILD SEQUENCES
# =========================
def build_sequences(df, max_len=MAX_LEN, is_train=True):
    sequences = []

    grouped = df.groupby(["doc", "page"])

    for _, group in grouped:
        group = group.sort_values("word_pos")

        words = group["word"].tolist()
        trt = group["answer"].values if is_train else None

        for i in range(0, len(words), max_len):
            chunk_words = words[i:i+max_len]

            if is_train:
                chunk_trt = trt[i:i+max_len]
                sequences.append((chunk_words, chunk_trt))
            else:
                sequences.append((chunk_words, None))

    return sequences


# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_sequences(sequences):
    encodings = []
    labels = []

    for words, trt in sequences:
        enc = tokenizer(
            words,
            is_split_into_words=True,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN
        )

        word_ids = enc.word_ids()

        if trt is not None:
            token_labels = []
            for wid in word_ids:
                if wid is None:
                    token_labels.append(-100)
                else:
                    token_labels.append(trt[wid])
            labels.append(torch.tensor(token_labels, dtype=torch.float32))

        encodings.append(enc)

    return encodings, labels


train_seq = build_sequences(train_df)
val_seq = build_sequences(val_df)

train_enc, train_labels = tokenize_sequences(train_seq)
val_enc, val_labels = tokenize_sequences(val_seq)


# =========================
# MODEL
# =========================
class TRTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(x).squeeze(-1)


model = TRTModel().to(DEVICE)


# =========================
# TRAINING
# =========================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def masked_mse(preds, target):
    mask = target != -100
    return ((preds[mask] - target[mask]) ** 2).mean()

def train_epoch(encs, labs):
    model.train()
    total_loss = 0

    for enc, lab in zip(encs, labs):
        optimizer.zero_grad()

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        lab = lab.unsqueeze(0).to(DEVICE)
        preds = preds.float()
        lab = lab.float()
        loss = masked_mse(preds, lab)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(encs)


def evaluate(encs, labs):
    model.eval()
    preds_all, y_all = [], []

    with torch.no_grad():
        for enc, lab in zip(encs, labs):
            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)

            preds = model(...).cpu().numpy()[0]
            lab = lab.numpy()

            mask = lab != -100
            preds_all.extend(preds[mask])
            y_all.extend(lab[mask])

    preds_all = np.array(preds_all)
    y_all = np.array(y_all)

    r2 = max(0, r2_score(y_all, preds_all))
    pear = abs(pearsonr(y_all, preds_all)[0])

    return 100 * (r2 + pear) / 2


# =========================
# TRAIN LOOP
# =========================
for epoch in range(5):
    loss = train_epoch(train_enc, train_labels)
    score = evaluate(val_enc, val_labels)
    print(f"Epoch {epoch} | Loss {loss:.4f} | Score {score:.4f}")


# =========================
# PREDICT TEST
# =========================
def predict_test(df):
    sequences = build_sequences(df, is_train=False)

    preds_all = []
    model.eval()

    with torch.no_grad():
        for words, _ in sequences:
            enc = tokenizer(
                words,
                is_split_into_words=True,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=MAX_LEN
            )

            word_ids = enc.word_ids()

            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)

            preds = model(input_ids, attention_mask).cpu().numpy()[0]

            word_preds, counts = {}, {}

            for p, wid in zip(preds, word_ids):
                if wid is None:
                    continue
                word_preds[wid] = word_preds.get(wid, 0) + p
                counts[wid] = counts.get(wid, 0) + 1

            final = [word_preds[i] / counts[i] for i in range(len(words))]
            preds_all.extend(final)

    return preds_all


preds = predict_test(test)

# inverse scaling
preds = scaler_y.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()


# =========================
# SUBMISSION
# =========================
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test["datapointID"],
    "answer": preds
})

submission.to_csv("submission.csv", index=False)

ImportError: tokenizers>=0.21,<0.22 is required for a normal functioning of this module, but found tokenizers==0.22.2.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main